# RAW-FABRID Swin-T held-out-roll reproduction

This notebook replays the study's existing frozen Swin-T and position-dependent Gaussian detector on Kaggle GPU. It starts with roll 3A and compares the saved output with the Mac run. Run all seven rolls only after the first run works. The network is pretrained and frozen. The normal Gaussian distribution is fitted for each held-out roll.

In Kaggle, enable a GPU and internet for downloading ImageNet weights. Attach the project code and the publisher ZIP as Kaggle inputs. The original ZIP must remain a ZIP and must match the project's expected SHA-256. If Kaggle automatically extracts your dataset archive, upload the ZIP inside a TAR archive and extract the TAR in the working directory first. Keep the notebook and inputs private if appropriate. Edit the two paths below. Never paste a GitHub token into the notebook.


In [ ]:
from pathlib import Path

# Project folder must contain code/, results/, and reports/.
SOURCE_PROJECT = Path('/kaggle/input/your-project/project-files')
# Path to the original 4.13 GB publisher ZIP, not an extracted directory.
RAW_ARCHIVE = Path('/kaggle/input/your-data/RAW_FABRID.zip')
# Rollo3A is the smallest fold. Change to None to run all seven rolls.
ROLL = 'Rollo3A'
WORK = Path('/kaggle/working/raw_fabrid_project')


In [ ]:
import importlib
import platform
import shutil
import sys
import torch
import torchvision

print('Python:', sys.version)
print('PyTorch:', torch.__version__, 'torchvision:', torchvision.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU under Notebook Settings and restart the session.')
print('GPU:', torch.cuda.get_device_name(0))
for package in ('numpy', 'scipy', 'sklearn', 'PIL'):
    importlib.import_module(package)
for name in ('code/unseen_roll.py', 'results/baseline_seed_13017.json',
             'results/localization_study.json', 'reports/unseen_roll_protocol.md'):
    if not (SOURCE_PROJECT / name).is_file():
        raise FileNotFoundError(f'Missing project file: {SOURCE_PROJECT / name}')
if not RAW_ARCHIVE.is_file():
    raise FileNotFoundError(f'Publisher ZIP not found: {RAW_ARCHIVE}')


In [ ]:
# Copy only the files needed for this replay. Existing committed results remain untouched.
WORK.mkdir(parents=True, exist_ok=True)
for folder in ('code', 'results', 'reports'):
    shutil.copytree(SOURCE_PROJECT / folder, WORK / folder, dirs_exist_ok=True)
(WORK / 'data').mkdir(exist_ok=True)
archive_link = WORK / 'data' / 'RAW_FABRID.zip'
if archive_link.is_symlink():
    archive_link.unlink()
elif archive_link.exists():
    raise FileExistsError(f'Remove or relocate the existing file: {archive_link}')
archive_link.symlink_to(RAW_ARCHIVE.resolve())
print('Project copy:', WORK)


In [ ]:
# Verify the actual archive before model fitting. This reads the whole ZIP.
sys.path.insert(0, str(WORK / 'code'))
from audit_zip import PUBLISHER_SHA256, sha256
actual = sha256(archive_link)
print('Actual SHA-256:  ', actual)
print('Expected SHA-256:', PUBLISHER_SHA256)
if actual != PUBLISHER_SHA256:
    raise ValueError('Archive does not match the version used for the original study.')


In [ ]:
# ImageNet weights download on first use. Keep the results separate from the Mac results.
import subprocess
output = Path('/kaggle/working/swin_replay')
command = [sys.executable, '-u', str(WORK / 'code' / 'unseen_roll.py'),
           '--model', 'swin_t', '--output-dir', str(output)]
if ROLL is not None:
    command.extend(['--roll', ROLL])
print('Running:', ' '.join(command))
subprocess.run(command, cwd=WORK, check=True)


In [ ]:
# Compare predictions to the Mac run. Different hardware can change floating-point values
# and move pixels across score bins. This is a diagnostic, not a pass/fail claim.
import json
import numpy as np
paths = sorted(output.glob('swin_t_Rollo*.json'))
if not paths:
    raise RuntimeError('No output files were written.')
for path in paths:
    new = json.loads(path.read_text())
    old = json.loads((WORK / 'results' / 'unseen_roll' / path.name).read_text())
    assert set(new['frames']) == set(old['frames'])
    assert new['ordered_tiles_sha256'] == old['ordered_tiles_sha256']
    assert new['weights_sha256'] == old['weights_sha256']
    differences = [abs(new['frames'][name]['scores'][policy] -
                       old['frames'][name]['scores'][policy])
                   for name in new['frames'] for policy in ('released', 'fixed')]
    assert all(new['frames'][name]['foreground'] == old['frames'][name]['foreground']
               and new['frames'][name]['unseen_foreground'] == old['frames'][name]['unseen_foreground']
               for name in new['frames'])
    print(path.name, '| frames:', len(new['frames']),
          '| max frame-score difference:', max(differences),
          '| GPU:', new['device'])
print('Download the result JSON files from /kaggle/working/swin_replay.')


The published comparison used both ResNet-18 and Swin-T. This notebook replays Swin-T only. To run both models or recompute all reported roll-level AP values, follow the project README and preserve the existing ResNet outputs. The GPU replay is a reproducibility check, not a new independent dataset or an independent confirmatory test.
